In [0]:
%pip install xgboost==2.0.3 scikit-learn==1.3.2 shap mlflow


In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.xgboost
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, lit, current_date, when
import shap


print("imports done")

**Load Production Model**

In [0]:
# load by alias — never hardcode version number
model_uri = "models:/bharatmart.ml.bharatmart_churn_model@Production"
model = mlflow.xgboost.load_model(model_uri)

print(f"model loaded from : {model_uri}")

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
version_info = client.get_model_version_by_alias(
    name="bharatmart.ml.bharatmart_churn_model",
    alias="Production"
)

print(f"model name : {version_info.name}")
print(f"model version : {version_info.version}")
print(f"run id : {version_info.run_id}")
print(f"status : {version_info.status}")

**Read Feature Table**

In [0]:
features_df = spark.table("bharatmart.ml.churn_features")

print(f"rows : {features_df.count():,}")
print(f"columns : {len(features_df.columns)}")

**Prepare Features for Scoring**

In [0]:
feature_cols = [
    "total_spent", "avg_order_value",
    "refund_count", "refund_rate", "abandon_rate",
    "avg_pages_viewed", "avg_session_duration", "session_count",
    "pct_organic", "pct_social", "avg_rating", "payment_idx"
]

from pyspark.sql.functions import col as scol

features_df = spark.table("bharatmart.ml.churn_features")

for c in ["total_spent", "avg_order_value", "refund_rate",
          "avg_pages_viewed", "avg_session_duration", "avg_rating"]:
    features_df = features_df.withColumn(c, scol(c).cast("double"))

score_df = features_df.select(["customer_id"] + feature_cols).toPandas()

print(f"scoring rows     : {score_df.shape[0]:,}")
print(f"scoring features : {score_df.shape[1] - 1}")

**Score All Customers**

In [0]:
# score all customers — get churn probability
churn_proba = model.predict_proba(score_df[feature_cols])[:,1]

score_df["churn_probability"] = churn_proba

print(f"min probability : {churn_proba.min():.4f}")
print(f"max probability : {churn_proba.max():.4f}")
print(f"mean probability : {churn_proba.mean():.4f}")

**Add Churn Risk Band**

In [0]:
# segment customers into risk bands based on churn probability
score_df["churn_risk_band"] = pd.cut(
    score_df["churn_probability"],
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=["Low", "Medium", "High", "Critical"],
    include_lowest=True
)

print(score_df["churn_risk_band"].value_counts().to_string())

**Add Top Churn Reason from SHAP**

In [0]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(score_df[feature_cols])

# get top feature per customer — highest absolute SHAP value
top_reason_idx = np.abs(shap_values).argmax(axis=1)
top_reason = [feature_cols[i] for i in top_reason_idx]

score_df["top_churn_reason"] = top_reason

print(pd.Series(top_reason).value_counts().to_string())

**Add Scored Date and Write Predictions**

In [0]:
from pyspark.sql.functions import current_date

# convert to spark dataframe
predictions_df = spark.createDataFrame(score_df[[
    "customer_id",
    "churn_probability",
    "churn_risk_band",
    "top_churn_reason"
]])

# add scored date
predictions_df = predictions_df.withColumn("scored_date", current_date())

# write to ml schema
predictions_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bharatmart.ml.gld_churn_predictions")

count = spark.table("bharatmart.ml.gld_churn_predictions").count()
print(f"wrote {count:,} rows to bharatmart.ml.gld_churn_predictions")

In [0]:
display(spark.table("bharatmart.ml.gld_churn_predictions").limit(10))